In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [ ]:
#hide
from fastbook import *
from IPython.display import display,HTML

# Глубокое погружение в область обработки естественного языка: Рекуррентные нейронные сети (RNN).

В разделе <<chapter_intro>> мы увидели, что глубокое обучение может давать отличные результаты при работе с наборами данных, содержащими естественный язык. В нашем примере мы использовали предварительно обученную языковую модель и дообучили ее для классификации отзывов. Этот пример подчеркнул разницу между переносом обучения в области обработки естественного языка (NLP) и компьютерного зрения: в общем случае, в NLP предварительно обученная модель обучается на другой задаче.

Языковая модель – это модель, которая обучена предсказывать следующее слово в тексте (исходя из прочитанных предыдущих слов). Этот тип задач называется *самообучением*: нам не нужно присваивать метки нашей модели, достаточно просто предоставить ей большое количество текстовых данных. В ней есть процесс автоматического получения меток из данных, и эта задача не является тривиальной: чтобы правильно предсказывать следующее слово в предложении, модель должна будет развить понимание английского (или другого) языка. Самообучение также может использоваться в других областях; например, ознакомьтесь со статьей ["Self-Supervised Learning and Computer Vision"](https://www.fast.ai/2020/01/13/self_supervised/) для ознакомления с приложениями в области компьютерного зрения. Самообучение обычно не используется для модели, которая обучается напрямую, а скорее используется для предварительного обучения модели, используемой для переноса обучения.

```markdown
> жаргон: Самообучение: Обучение модели с использованием меток, которые содержатся в независимой переменной, а не требующих внешних меток. Например, обучение модели для предсказания следующего слова в тексте.
```

В главе <<chapter_intro>> мы использовали языковую модель, предварительно обученную на Wikipedia, для классификации отзывов с сайта IMDb. Мы получили отличные результаты, непосредственно настроив эту языковую модель для классификатора отзывов о фильмах, но, добавив еще один этап, мы можем добиться еще лучших результатов. Английский язык, используемый на Wikipedia, немного отличается от языка, используемого на IMDb, поэтому вместо того, чтобы сразу переходить к классификатору, мы могли бы предварительно настроить нашу предварительно обученную языковую модель на корпусе IMDb, а затем использовать *ее* в качестве основы для нашего классификатора.

Даже если наша языковая модель знает основы языка, который мы используем в задаче (например, наша предварительно обученная модель работает на английском языке), полезно привыкнуть к стилю целевого корпуса. Это может быть более неформальный язык или более технический, с новыми словами для изучения или различными способами построения предложений. В случае набора данных IMDb будет много имен режиссеров и актеров, а также, как правило, менее формальный стиль языка, чем на Wikipedia.

Мы уже видели, что с помощью fastai можно загрузить предварительно обученную английскую языковую модель и использовать ее для достижения передовых результатов в задачах классификации в области обработки естественного языка. (Мы ожидаем, что в ближайшее время станет доступно гораздо больше предварительно обученных моделей на разных языках – возможно, они будут доступны к тому моменту, когда вы читаете эту книгу). Итак, зачем мы изучаем, как обучать языковую модель в деталях?

Одна из причин, конечно, заключается в том, что полезно понимать основы моделей, которые вы используете. Но есть еще одна очень практичная причина: вы получаете еще лучшие результаты, если предварительно настроить языковую модель (основанную на последовательностях), прежде чем настраивать классификационную модель. Например, для задачи анализа тональности отзывов о фильмах на IMDb, набор данных включает 50 000 дополнительных отзывов о фильмах, которые не имеют положительных или отрицательных меток. Поскольку в обучающем наборе есть 25 000 отмеченных отзывов, а в наборе для проверки – 25 000, всего получается 100 000 отзывов о фильмах. Мы можем использовать все эти отзывы для предварительной настройки предварительно обученной языковой модели, которая была обучена только на статьях Wikipedia; это приведет к созданию языковой модели, которая особенно хорошо предсказывает следующее слово в отзыве о фильме.

Этот подход известен как Universal Language Model Fine-tuning (ULMFit). В [статье](https://arxiv.org/abs/1801.06146) показано, что этот дополнительный этап предварительной настройки языковой модели, предшествующий переносу обучения на задачу классификации, приводит к значительно более точным результатам. Используя этот подход, у нас есть три этапа переноса обучения в области обработки естественного языка, как это показано в <<ulmfit_process>>.


```markdown
<img alt="Схема процесса ULMFiT" width="700" caption="Процесс ULMFiT" id="ulmfit_process" src="images/att_00027.png">
```

Теперь мы рассмотрим, как можно применить нейронную сеть для решения этой задачи моделирования языка, используя концепции, представленные в двух предыдущих главах. Но прежде чем читать дальше, остановитесь и подумайте о том, как *вы* бы подошли к этой задаче.

## Предварительная обработка текста


Совершенно не очевидно, как мы будем использовать полученные знания для создания языковой модели. Предложения могут иметь разную длину, а документы – быть очень длинными. Так как же мы можем предсказать следующее слово в предложении, используя нейронную сеть? Давайте выясним!

Мы уже видели, как категориальные переменные можно использовать в качестве независимых переменных для нейронной сети. Подход, который мы использовали для одной категориальной переменной, заключался в следующем:

1. Составить список всех возможных значений этой категориальной переменной (мы будем называть этот список *словарем*).
2. Заменить каждое значение его индексом в словаре.
3. Создать матрицу вложений (embedding matrix), содержащую строку для каждого значения (то есть, для каждого элемента словаря).
4. Использовать эту матрицу вложений в качестве первого слоя нейронной сети. (Специальная матрица вложений может принимать на вход необработанные индексы, созданные на шаге 2; это эквивалентно, но быстрее и эффективнее, чем матрица, которая принимает на вход векторы, представляющие индексы в формате "one-hot").

Мы можем сделать почти то же самое с текстом! Новое здесь – это концепция последовательности. Сначала мы объединяем все документы из нашего набора данных в одну большую строку и разбиваем ее на слова, получая очень длинный список слов (или "токенов"). Наша независимая переменная будет представлять собой последовательность слов, начинающуюся с первого слова в нашем длинном списке и заканчивающуюся предпоследним, а зависимая переменная – последовательность слов, начинающаяся со второго слова и заканчивающаяся последним.

Наш словарь будет состоять из смеси распространенных слов, которые уже есть в словаре нашей предварительно обученной модели, и новых слов, специфичных для нашего корпуса (например, кинотерминов или имен актеров). Наша матрица вложений будет построена соответствующим образом: для слов, которые есть в словаре нашей предварительно обученной модели, мы возьмем соответствующую строку из матрицы вложений этой модели; но для новых слов у нас ничего не будет, поэтому мы просто инициализируем соответствующую строку случайным вектором.


Каждый из этапов, необходимых для создания языковой модели, связан с терминологией из области обработки естественного языка, и библиотеки fastai и PyTorch предоставляют классы, которые могут помочь. Этапы включают в себя:

- Токенизация: преобразование текста в список слов (или символов, или подстрок, в зависимости от уровня детализации вашей модели).
- Нумеризация: создание списка всех уникальных слов (словаря) и преобразование каждого слова в число, путем поиска его индекса в словаре.
- Создание загрузчика данных для языковой модели: fastai предоставляет класс `LMDataLoader`, который автоматически создает зависимую переменную, которая отличается от независимой переменной на один токен. Он также обрабатывает некоторые важные детали, такие как то, как перемешивать обучающие данные таким образом, чтобы зависимая и независимая переменные сохраняли свою структуру, как это требуется.
- Создание языковой модели: нам нужна специальная модель, которая выполняет то, чего мы раньше не видели: она должна обрабатывать входные списки, которые могут быть произвольно большими или маленькими. Существует несколько способов сделать это; в этой главе мы будем использовать *рекуррентную нейронную сеть* (RNN). Мы рассмотрим детали этих RNN в главе <<chapter_nlp_dive>>, но пока вы можете думать о ней просто как о еще одной глубокой нейронной сети.

Давайте подробно рассмотрим, как работает каждый этап.

### Токенизация


Когда мы говорили "преобразовать текст в список слов", мы упустили много деталей. Например, что делать с пунктуацией? Как поступать со словами, такими как "don't"? Это одно слово или два? Что насчет длинных медицинских или химических слов? Следует ли их разбивать на отдельные смысловые части? А как насчет слов, содержащих дефис? Что делать с такими языками, как немецкий и польский, где можно создавать очень длинные слова, состоящие из множества частей? А как насчет языков, таких как японский и китайский, которые вообще не используют корневые слова и не имеют четкого представления о том, что такое *слово*?

Поскольку на эти вопросы нет однозначного ответа, не существует единого подхода к токенизации. Существуют три основных подхода:

- На основе слов: Разделение предложения на части по пробелам, а также применение правил, специфичных для каждого языка, для попытки разделения смысловых частей, даже если между ними нет пробелов (например, преобразование "don't" в "do n't"). Обычно знаки препинания также разбиваются на отдельные токены.
- На основе подслов: Разделение слов на более мелкие части, основанное на наиболее часто встречающихся подстроках. Например, слово "occasion" может быть токенизировано как "o c ca sion".
- На основе символов: Разделение предложения на отдельные символы.

В этом разделе мы рассмотрим токенизацию на основе слов и подслов, а токенизацию на основе символов мы оставим вам для реализации в анкете в конце этой главы.

```markdown
> жаргон: токен: Один из элементов списка, созданного в процессе токенизации. Это может быть слово, часть слова (подслово) или отдельный символ.
```

### Токенизация слов с использованием библиотеки fastai


Вместо того чтобы предоставлять собственные инструменты для токенизации, библиотека fastai предлагает унифицированный интерфейс для работы с различными инструментами токенизации, доступными в сторонних библиотеках. Токенизация – это активно развивающаяся область исследований, и постоянно появляются новые и улучшенные инструменты, поэтому параметры по умолчанию, используемые в fastai, также могут меняться. Однако, API и опции не должны сильно меняться, поскольку fastai стремится поддерживать стабильный API, даже если меняются базовые технологии.

Давайте попробуем это на примере набора данных IMDb, который мы использовали в разделе <<chapter_intro>>:

In [ ]:
from fastai.text.all import *
path = untar_data(URLs.IMDB)

Нам нужно получить текстовые файлы, чтобы попробовать использовать токенизатор. Как и функция `get_image_files`, которую мы уже много раз использовали, которая получает все файлы изображений в указанном пути, функция `get_text_files` получает все текстовые файлы в указанном пути. Также, по желанию, можно передать параметр `folders`, чтобы ограничить поиск определенным списком подпапок:

In [ ]:
files = get_text_files(path, folders = ['train', 'test', 'unsup'])

Вот один из отзывов, который мы будем анализировать (мы приведем здесь только начало, чтобы сэкономить место):


In [ ]:
txt = files[0].open().read(); txt[:75]

'This movie, which I just discovered at the video store, has apparently sit '

В процессе написания этой книги, библиотека *spaCy* используется в качестве стандартного модуля для токенизации английского текста в fastai. Она обладает сложным механизмом обработки правил, включающим специальные правила для URL-адресов, отдельных специальных английских слов и многого другого. Однако, вместо прямого использования `SpacyTokenizer`, мы будем использовать `WordTokenizer`, поскольку он всегда будет указывать на текущий стандартный токенизатор слов, используемый fastai (который может быть не spaCy, в зависимости от того, когда вы читаете это).

Давайте попробуем. Мы будем использовать функцию `coll_repr(collection, n)` из fastai для отображения результатов. Эта функция отображает первые *n* элементов из *collection*, а также общий размер – это то, что `L` использует по умолчанию. Обратите внимание, что токенизаторы fastai принимают коллекцию документов для токенизации, поэтому нам нужно обернуть `txt` в список:


In [ ]:
spacy = WordTokenizer()
toks = first(spacy([txt]))
print(coll_repr(toks, 30))

(#201) ['This','movie',',','which','I','just','discovered','at','the','video','store',',','has','apparently','sit','around','for','a','couple','of','years','without','a','distributor','.','It',"'s",'easy','to','see'...]


Как вы видите, spaCy в основном просто разделяет слова и знаки препинания. Но здесь она также делает кое-что еще: она разбивает "it's" на "it" и "'s". Это имеет логическое объяснение, поскольку это, по сути, отдельные слова. Токенизация – это удивительно тонкая задача, если подумать обо всех мелких деталях, которые необходимо учитывать. К счастью, spaCy хорошо справляется с этим для нас. Например, здесь мы видим, что "." разделяется, когда он завершает предложение, но не в аббревиатуре или числе:

In [ ]:
first(spacy(['The U.S. dollar $1 is $1.00.']))

(#9) ['The','U.S.','dollar','$','1','is','$','1.00','.']

Библиотека fastai добавляет дополнительные функции к процессу токенизации с помощью класса `Tokenizer`:

In [ ]:
tkn = Tokenizer(spacy)
print(coll_repr(tkn(txt), 31))

(#228) ['xxbos','xxmaj','this','movie',',','which','i','just','discovered','at','the','video','store',',','has','apparently','sit','around','for','a','couple','of','years','without','a','distributor','.','xxmaj','it',"'s",'easy'...]


Обратите внимание, что теперь появились некоторые токены, начинающиеся с символов "xx", что не является распространенным префиксом слов в английском языке. Это *специальные токены*.

Например, первый элемент в списке, `xxbos`, является специальным токеном, который указывает на начало нового текста ("BOS" - стандартный аббревиатурный термин в области обработки естественного языка, означающий "начало потока"). Распознавая этот начальный токен, модель сможет понять, что ей необходимо "забыть" то, что было сказано ранее, и сосредоточиться на предстоящих словах.

Эти специальные токены не поступают напрямую из библиотеки spaCy. Они добавляются библиотекой fastai по умолчанию, применяя ряд правил при обработке текста. Эти правила предназначены для того, чтобы модели было легче распознавать важные части предложения. В некотором смысле, мы преобразуем исходную последовательность слов на английском языке в упрощенный токенизированный язык — язык, предназначенный для того, чтобы модель могла его легко изучить.

Например, правила заменят последовательность из четырех восклицательных знаков специальным токеном, обозначающим *повторяющийся символ*, за которым следует число четыре, а затем один восклицательный знак. Таким образом, матрица представлений модели может кодировать информацию об общих понятиях, таких как повторные знаки препинания, вместо того, чтобы требовать отдельный токен для каждого количества повторений каждого знака препинания. Аналогично, слово, начинающееся с заглавной буквы, будет заменено специальным токеном, обозначающим заглавную букву, за которым следует версия этого слова в нижнем регистре. Таким образом, матрице представлений нужны только версии слов в нижнем регистре, что экономит вычислительные ресурсы и память, но при этом модель может изучить концепцию заглавных букв.

Вот некоторые из основных специальных токенов, которые вы будете видеть:

- `xxbos`: Указывает на начало текста (в данном случае, отзыва).
- `xxmaj`: Указывает на то, что следующее слово начинается с заглавной буквы (поскольку все было преобразовано в нижний регистр).
- `xxunk`: Указывает на то, что слово неизвестно.

Чтобы увидеть правила, которые были использованы, вы можете проверить правила по умолчанию:


In [ ]:
defaults.text_proc_rules

[<function fastai.text.core.fix_html(x)>,
 <function fastai.text.core.replace_rep(t)>,
 <function fastai.text.core.replace_wrep(t)>,
 <function fastai.text.core.spec_add_spaces(t)>,
 <function fastai.text.core.rm_useless_spaces(t)>,
 <function fastai.text.core.replace_all_caps(t)>,
 <function fastai.text.core.replace_maj(t)>,
 <function fastai.text.core.lowercase(t, add_bos=True, add_eos=False)>]

Как обычно, вы можете посмотреть исходный код каждой из этих функций в блокноте, введя:

```
??replace_rep
```

Вот краткое описание того, что делает каждая из них:

- `fix_html`: Заменяет специальные HTML-символы на удобочитаемые версии (в отзывах IMDb их довольно много).
- `replace_rep`: Заменяет любой символ, который повторяется три или более раза, на специальный токен, обозначающий повторение (`xxrep`), затем указывает количество повторений, а затем сам символ.
- `replace_wrep`: Заменяет любое слово, которое повторяется три или более раза, на специальный токен, обозначающий повторение слова (`xxwrep`), затем указывает количество повторений, а затем само слово.
- `spec_add_spaces`: Добавляет пробелы вокруг символов "/" и "#".
- `rm_useless_spaces`: Удаляет все повторения символа пробела.
- `replace_all_caps`: Переводит слово, написанное заглавными буквами, в нижний регистр и добавляет специальный токен, обозначающий использование заглавных букв (`xxup`), перед ним.
- `replace_maj`: Переводит слово, написанное с заглавной буквы, в нижний регистр и добавляет специальный токен, обозначающий использование заглавной буквы (`xxmaj`), перед ним.
- `lowercase`: Переводит весь текст в нижний регистр и добавляет специальный токен в начале (`xxbos`) и/или в конце (`xxeos`).

Давайте рассмотрим несколько примеров их работы:

In [ ]:
coll_repr(tkn('&copy;   Fast.ai www.fast.ai/INDEX'), 31)

"(#11) ['xxbos','©','xxmaj','fast.ai','xxrep','3','w','.fast.ai','/','xxup','index'...]"

Теперь давайте рассмотрим, как бы работала токенизация на уровне подслов.

### Токенизация на уровне подслов.


Помимо подхода к токенизации, основанного на разделении по словам, который был рассмотрен в предыдущем разделе, существует еще один популярный метод – токенизация по подсловам. Токенизация по словам предполагает, что пробелы обеспечивают полезное разделение компонентов смысла в предложении. Однако это предположение не всегда верно. Например, рассмотрим это предложение: 我的名字是郝杰瑞 ("Мое имя – Jeremy Howard" на китайском языке). Это предложение плохо подойдет для токенизатора, работающего со словами, потому что в нем нет пробелов! В таких языках, как китайский и японский, не используются пробелы, и, фактически, у них даже нет четкого понятия "слова". Существуют также языки, такие как турецкий и венгерский, в которых можно объединять множество подслов без использования пробелов, создавая очень длинные слова, содержащие большое количество отдельных фрагментов информации.

Для обработки таких случаев, как правило, лучше использовать токенизацию по подсловам. Этот процесс состоит из двух этапов:

1. Анализ корпуса документов для выявления наиболее часто встречающихся групп букв. Эти группы становятся словарным запасом.
2. Токенизация корпуса с использованием этого словарного запаса, состоящего из *подсловных единиц*.

Рассмотрим пример. Для нашего корпуса мы будем использовать первые 2000 отзывов о фильмах:


In [ ]:
txts = L(o.open().read() for o in files[:2000])

Мы инициализируем наш токенизатор, передавая ему размер словаря, который мы хотим создать, а затем нам необходимо "обучить" его. То есть, нам нужно, чтобы он прочитал наши документы и нашел наиболее часто встречающиеся последовательности символов для создания этого словаря. Это делается с помощью функции `setup`. Как мы увидим позже, `setup` – это специальный метод fastai, который автоматически вызывается в наших обычных конвейерах обработки данных. Однако, поскольку мы сейчас выполняем все вручную, нам необходимо вызывать его самостоятельно. Вот функция, которая выполняет эти шаги для заданного размера словаря, и показывает пример вывода:


In [ ]:
def subword(sz):
    sp = SubwordTokenizer(vocab_sz=sz)
    sp.setup(txts)
    return ' '.join(first(sp([txt]))[:40])

Давайте попробуем:


In [ ]:
subword(1000)

'▁This ▁movie , ▁which ▁I ▁just ▁dis c over ed ▁at ▁the ▁video ▁st or e , ▁has ▁a p par ent ly ▁s it ▁around ▁for ▁a ▁couple ▁of ▁years ▁without ▁a ▁dis t ri but or . ▁It'

При использовании токенизатора подслов от fastai, специальный символ ` ` представляет собой пробел в исходном тексте.

Если используется более ограниченный словарь, то каждый токен будет соответствовать меньшему количеству символов, и для представления предложения потребуется больше токенов:


In [ ]:
subword(200)

'▁ T h i s ▁movie , ▁w h i ch ▁I ▁ j us t ▁ d i s c o ver ed ▁a t ▁the ▁ v id e o ▁ st or e , ▁h a s'

С другой стороны, если мы используем более обширный словарь, то наиболее распространенные английские слова, скорее всего, окажутся в самом словаре, и нам не потребуется так много слов для представления предложения:

In [ ]:
subword(10000)

"▁This ▁movie , ▁which ▁I ▁just ▁discover ed ▁at ▁the ▁video ▁store , ▁has ▁apparently ▁sit ▁around ▁for ▁a ▁couple ▁of ▁years ▁without ▁a ▁distributor . ▁It ' s ▁easy ▁to ▁see ▁why . ▁The ▁story ▁of ▁two ▁friends ▁living"

Выбор размера словаря подслов – это компромисс: больший словарь означает меньше токенов в предложении, что приводит к более быстрой тренировке, меньшему потреблению памяти и меньшему объему информации, которую модели необходимо запоминать. Однако, это также означает более крупные матрицы эмбеддингов, для обучения которых требуется больше данных.

В целом, токенизация на основе подслов предоставляет возможность легко масштабировать систему между токенизацией на уровне символов (то есть, с использованием небольшого словаря подслов) и токенизацией на уровне слов (то есть, с использованием большого словаря подслов), и позволяет обрабатывать любой человеческий язык без необходимости разработки алгоритмов, специфичных для каждого языка. Она может даже обрабатывать другие "языки", такие как геномные последовательности или нотации MIDI! Именно поэтому, за последний год ее популярность резко возросла, и, вероятно, она станет наиболее распространенным подходом к токенизации (возможно, она уже им является к моменту, когда вы это читаете!).

Как только наши тексты будут разбиты на отдельные элементы (токены), нам необходимо преобразовать их в числовые значения. Мы рассмотрим этот процесс далее.

### Преобразование данных в числовой формат с использованием библиотеки fastai.


*Численное представление* – это процесс сопоставления токенов с целыми числами. Шаги, необходимые для этого процесса, в основном идентичны шагам, необходимым для создания переменной типа "категориальная", например, переменной, представляющей цифры в наборе данных MNIST:

1. Составьте список всех возможных значений этой категориальной переменной (словарь).
2. Замените каждое значение его индексом в словаре.

Давайте рассмотрим это на примере токенизированного текста, который мы видели ранее:

In [ ]:
toks = tkn(txt)
print(coll_repr(tkn(txt), 31))

(#228) ['xxbos','xxmaj','this','movie',',','which','i','just','discovered','at','the','video','store',',','has','apparently','sit','around','for','a','couple','of','years','without','a','distributor','.','xxmaj','it',"'s",'easy'...]


Как и в случае с `SubwordTokenizer`, нам необходимо вызвать функцию `setup` для объекта `Numericalize`; именно так мы создаем словарь. Это означает, что нам сначала потребуется токенизированный корпус. Поскольку токенизация занимает некоторое время, fastai выполняет ее параллельно; однако, для этого пошагового руководства мы будем использовать небольшой фрагмент данных:

In [ ]:
toks200 = txts[:200].map(tkn)
toks200[0]

(#228) ['xxbos','xxmaj','this','movie',',','which','i','just','discovered','at'...]

Мы можем передать это в функцию `setup` для создания нашего словаря:


In [ ]:
num = Numericalize()
num.setup(toks200)
coll_repr(num.vocab,20)

"(#2000) ['xxunk','xxpad','xxbos','xxeos','xxfld','xxrep','xxwrep','xxup','xxmaj','the','.',',','a','and','of','to','is','in','i','it'...]"

Наши специальные токены правил отображаются первыми, а затем каждое слово появляется один раз, в порядке убывания частоты. Значения по умолчанию для функции `Numericalize` – `min_freq=3` и `max_vocab=60000`. Установка `max_vocab=60000` приводит к тому, что библиотека fastai заменяет все слова, кроме 60 000 самых распространенных, специальным токеном "*неизвестное слово*", `xxunk`. Это полезно, чтобы избежать создания слишком большой матрицы эмбеддингов, так как это может замедлить процесс обучения и потреблять слишком много памяти. Кроме того, это может означать, что недостаточно данных для обучения полезных представлений для редких слов. Однако, эту последнюю проблему лучше решать, устанавливая значение `min_freq`; значение по умолчанию `min_freq=3` означает, что любое слово, которое встречается менее трех раз, заменяется токеном `xxunk`.

Fastai также может преобразовать ваш набор данных в числовой формат, используя словарь, который вы предоставляете, передавая список слов в качестве параметра `vocab`.

После создания объекта `Numericalize` мы можем использовать его, как если бы это была функция:


In [ ]:
nums = num(toks)[:20]; nums

tensor([  2,   8,  21,  28,  11,  90,  18,  59,   0,  45,   9, 351, 499,  11,  72, 533, 584, 146,  29,  12])

В этот раз наши токены были преобразованы в тензор целых чисел, который может быть получен нашей моделью. Мы можем проверить, что они соответствуют исходному тексту:


In [ ]:
' '.join(num.vocab[o] for o in nums)

'xxbos xxmaj this movie , which i just xxunk at the video store , has apparently sit around for a'

Теперь, когда у нас есть данные, нам нужно разделить их на группы для обучения нашей модели.

### Разбиение наших текстовых данных на пакеты для использования в языковой модели.

При работе с изображениями нам необходимо было изменить размер всех изображений, чтобы они имели одинаковую высоту и ширину, прежде чем объединять их в мини-пакеты, чтобы они могли эффективно располагаться в одном тензоре. Здесь ситуация будет немного иной, поскольку текст нельзя просто масштабировать до нужной длины. Кроме того, мы хотим, чтобы наша языковая модель читала текст последовательно, чтобы она могла эффективно предсказывать следующее слово. Это означает, что каждая новая партия должна начинаться точно там, где закончилась предыдущая.

Предположим, у нас есть следующий текст:

> : В этой главе мы вернемся к примеру классификации отзывов о фильмах, который мы изучали в главе 1, и рассмотрим его более подробно. Сначала мы рассмотрим этапы обработки, необходимые для преобразования текста в числа, и способы его настройки. Таким образом, мы получим еще один пример использования класса PreProcessor в API для работы с данными.
> Затем мы изучим, как мы создаем языковую модель и обучаем ее в течение некоторого времени.

Процесс токенизации добавит специальные символы и обработает знаки препинания, чтобы получить следующий текст:

> : xxbos xxmaj in this chapter , we will go back over the example of classifying movie reviews we studied in chapter 1 and dig deeper under the surface . xxmaj first we will look at the processing steps necessary to convert text into numbers and how to customize it . xxmaj by doing this , we 'll have another example of the preprocessor used in the data block xxup api . \n xxmaj then we will study how we build a language model and train it for a while .

Теперь у нас есть 90 токенов, разделенных пробелами. Предположим, мы хотим размер пакета, равный 6. Нам нужно разделить этот текст на 6 последовательных частей длиной 15:


In [ ]:
#hide_input
stream = "In this chapter, we will go back over the example of classifying movie reviews we studied in chapter 1 and dig deeper under the surface. First we will look at the processing steps necessary to convert text into numbers and how to customize it. By doing this, we'll have another example of the PreProcessor used in the data block API.\nThen we will study how we build a language model and train it for a while."
tokens = tkn(stream)
bs,seq_len = 6,15
d_tokens = np.array([tokens[i*seq_len:(i+1)*seq_len] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

xxbos,xxmaj,in,this,chapter,",",we,will,go,back,over,the,example,of,classifying
movie,reviews,we,studied,in,chapter,1,and,dig,deeper,under,the,surface,.,xxmaj
first,we,will,look,at,the,processing,steps,necessary,to,convert,text,into,numbers,and
how,to,customize,it,.,xxmaj,by,doing,this,",",we,'ll,have,another,example
of,the,preprocessor,used,in,the,data,block,xxup,api,.,\n,xxmaj,then,we
will,study,how,we,build,a,language,model,and,train,it,for,a,while,.


В идеальном мире мы могли бы передать этот пакет данных нашей модели. Однако, такой подход не масштабируется, потому что, за пределами этого простого примера, маловероятно, что один пакет, содержащий все тексты, поместится в память нашей видеокарты (здесь у нас 90 токенов, а все отзывы IMDb вместе составляют несколько миллионов).

Поэтому, нам необходимо разделить этот массив на более мелкие подмассивы фиксированной длины. Важно сохранить порядок внутри и между этими подмассивами, потому что мы будем использовать модель, которая поддерживает состояние, чтобы она помнила, что было прочитано ранее, при прогнозировании того, что последует дальше.

Вернемся к нашему предыдущему примеру с 6 пакетами по 15 элементов. Если мы выберем длину последовательности равной 5, то мы сначала передадим следующую последовательность:


In [ ]:
#hide_input
bs,seq_len = 6,5
d_tokens = np.array([tokens[i*15:i*15+seq_len] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

xxbos,xxmaj,in,this,chapter
movie,reviews,we,studied,in
first,we,will,look,at
how,to,customize,it,.
of,the,preprocessor,used,in
will,study,how,we,build


Затем этот:


In [ ]:
#hide_input
bs,seq_len = 6,5
d_tokens = np.array([tokens[i*15+seq_len:i*15+2*seq_len] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

",",we,will,go,back
chapter,1,and,dig,deeper
the,processing,steps,necessary,to
xxmaj,by,doing,this,","
the,data,block,xxup,api
a,language,model,and,train


И, наконец:


In [ ]:
#hide_input
bs,seq_len = 6,5
d_tokens = np.array([tokens[i*15+10:i*15+15] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

over,the,example,of,classifying
under,the,surface,.,xxmaj
convert,text,into,numbers,and
we,'ll,have,another,example
.,\n,xxmaj,then,we
it,for,a,while,.


Возвращаясь к нашему набору данных с отзывами о фильмах, первый шаг – преобразовать отдельные текстовые фрагменты в единый поток, объединив их. Как и в случае с изображениями, лучше рандомизировать порядок входных данных. Поэтому в начале каждой эпохи мы перемешиваем записи, чтобы создать новый поток (мы перемешиваем порядок документов, а не порядок слов внутри них, иначе тексты потеряют смысл!).

Затем мы разбиваем этот поток на определенное количество пакетов (это и есть наш *размер пакета*). Например, если в потоке 50 000 токенов, и мы устанавливаем размер пакета равным 10, то у нас получится 10 мини-потоков по 5000 токенов. Важно, чтобы мы сохраняли порядок токенов (например, от 1 до 5000 для первого мини-потока, затем от 5001 до 10000...), потому что мы хотим, чтобы модель читала непрерывные последовательности текста (как в предыдущем примере). Токен `xxbos` добавляется в начало каждого мини-потока на этапе предобработки, чтобы модель знала, когда начинается новый фрагмент в потоке.

Итак, чтобы резюмировать, на каждой эпохе мы перемешиваем нашу коллекцию документов и объединяем их в поток токенов. Затем мы разбиваем этот поток на пакеты, состоящие из последовательных мини-потоков фиксированного размера. Наша модель затем будет читать мини-потоки в порядке, и благодаря внутреннему состоянию, она будет выдавать одинаковые результаты активации независимо от выбранной длины последовательности.

Все это происходит в фоновом режиме библиотекой fastai, когда мы создаем объект `LMDataLoader`. Мы делаем это, сначала применяя наш объект `Numericalize` к токенизированным текстам:


In [ ]:
nums200 = toks200.map(num)

и затем передавая эти данные в `LMDataLoader`:

In [ ]:
dl = LMDataLoader(nums200)

Давайте убедимся, что это дает ожидаемые результаты, взяв первую партию:


In [ ]:
x,y = first(dl)
x.shape,y.shape

(torch.Size([64, 72]), torch.Size([64, 72]))

затем обратите внимание на первую строку независимой переменной, которая должна соответствовать началу первого текста:

In [ ]:
' '.join(num.vocab[o] for o in x[0][:20])

'xxbos xxmaj this movie , which i just xxunk at the video store , has apparently sit around for a'

Зависимая переменная – это та же самая сущность, но сдвинутая на один токен:


In [ ]:
' '.join(num.vocab[o] for o in y[0][:20])

'xxmaj this movie , which i just xxunk at the video store , has apparently sit around for a couple'

Это завершает все этапы предварительной обработки, которые нам необходимо выполнить для наших данных. Теперь мы готовы обучить нашу систему классификации текста.

## Обучение текстового классификатора


Как мы видели в начале этой главы, процесс обучения современной системы классификации текста с использованием переноса обучения состоит из двух этапов: сначала необходимо адаптировать нашу языковую модель, предварительно обученную на данных Википедии, к корпусу отзывов IMDb, а затем использовать эту модель для обучения классификатора.

Как обычно, давайте начнем со сбора данных.

### Языковая модель, использующая DataBlock.

Библиотека fastai автоматически выполняет токенизацию и преобразование в числовые значения, когда объект `TextBlock` передается в `DataBlock`. Все аргументы, которые можно передать функциям `Tokenize` и `Numericalize`, также можно передать объекту `TextBlock`. В следующей главе мы рассмотрим самые простые способы выполнения каждого из этих этапов по отдельности, что облегчит отладку. Однако вы всегда можете отладить код, выполняя их вручную на небольшом наборе ваших данных, как показано в предыдущих разделах. И не забывайте об удобном методе `summary` объекта `DataBlock`, который очень полезен для отладки проблем с данными.

Вот как мы используем `TextBlock` для создания языковой модели, используя значения по умолчанию библиотеки fastai:


In [ ]:
get_imdb = partial(get_text_files, folders=['train', 'test', 'unsup'])

dls_lm = DataBlock(
    blocks=TextBlock.from_folder(path, is_lm=True),
    get_items=get_imdb, splitter=RandomSplitter(0.1)
).dataloaders(path, path=path, bs=128, seq_len=80)

Одно из отличий от предыдущих типов, которые мы использовали в `DataBlock`, заключается в том, что мы не используем класс напрямую (например, `TextBlock(...)`), а вызываем *метод класса*. Метод класса – это метод в Python, который, как следует из названия, принадлежит *классу*, а не *объекту*. (Обязательно поищите дополнительную информацию о методах класса в интернете, если вы с ними не знакомы, поскольку они широко используются во многих библиотеках и приложениях Python; мы уже несколько раз использовали их ранее в этой книге, но не акцентировали на этом внимание). `TextBlock` является особенным тем, что создание словаря для числового представления (numericalizer) может занять много времени (нам нужно прочитать и токенизировать каждый документ, чтобы получить этот словарь). Чтобы обеспечить максимальную эффективность, он выполняет несколько оптимизаций:

- Он сохраняет токенизированные документы во временной папке, чтобы не приходилось токенизировать их более одного раза.
- Он выполняет несколько процессов токенизации параллельно, чтобы использовать процессоры вашего компьютера.

Нам нужно указать `TextBlock`, как получить доступ к текстам, чтобы он мог выполнить эту начальную предобработку – это и делает функция `from_folder`.

Затем `show_batch` работает в обычном режиме:


In [ ]:
dls_lm.show_batch(max_n=2)

,text,text_
0,"xxbos xxmaj it 's awesome ! xxmaj in xxmaj story xxmaj mode , your going from punk to pro . xxmaj you have to complete goals that involve skating , driving , and walking . xxmaj you create your own skater and give it a name , and you can make it look stupid or realistic . xxmaj you are with your friend xxmaj eric throughout the game until he betrays you and gets you kicked off of the skateboard","xxmaj it 's awesome ! xxmaj in xxmaj story xxmaj mode , your going from punk to pro . xxmaj you have to complete goals that involve skating , driving , and walking . xxmaj you create your own skater and give it a name , and you can make it look stupid or realistic . xxmaj you are with your friend xxmaj eric throughout the game until he betrays you and gets you kicked off of the skateboard xxunk"
1,"what xxmaj i 've read , xxmaj death xxmaj bed is based on an actual dream , xxmaj george xxmaj barry , the director , successfully transferred dream to film , only a genius could accomplish such a task . \n\n xxmaj old mansions make for good quality horror , as do portraits , not sure what to make of the killer bed with its killer yellow liquid , quite a bizarre dream , indeed . xxmaj also , this","xxmaj i 've read , xxmaj death xxmaj bed is based on an actual dream , xxmaj george xxmaj barry , the director , successfully transferred dream to film , only a genius could accomplish such a task . \n\n xxmaj old mansions make for good quality horror , as do portraits , not sure what to make of the killer bed with its killer yellow liquid , quite a bizarre dream , indeed . xxmaj also , this is"


Теперь, когда наши данные готовы, мы можем выполнить тонкую настройку предварительно обученной языковой модели.

### Тонкая настройка языковой модели.

Чтобы преобразовать целочисленные индексы слов в векторы, которые мы можем использовать в нашей нейронной сети, мы будем использовать эмбеддинги, как мы делали это для рекомендательных систем и моделей, основанных на табличных данных. Затем мы подадим эти эмбеддинги в *рекуррентную нейронную сеть* (RNN), используя архитектуру под названием *AWD-LSTM* (мы покажем, как создать такую модель с нуля в главе <<chapter_nlp_dive>>). Как мы уже обсуждали, эмбеддинги в предварительно обученной модели объединяются со случайными эмбеддингами, добавленными для слов, которые отсутствовали в словаре, используемом при предварительном обучении. Это происходит автоматически внутри класса `language_model_learner`:


In [ ]:
learn = language_model_learner(
    dls_lm, AWD_LSTM, drop_mult=0.3, 
    metrics=[accuracy, Perplexity()]).to_fp16()

Функция потерь, используемая по умолчанию, – это кросс-энтропия, поскольку, по сути, мы имеем дело с задачей классификации (различные категории – это слова в нашем словаре). Метрика *perplexity* (неопределенность), используемая здесь, часто применяется в области обработки естественного языка для языковых моделей: это экспонента от функции потерь, то есть `torch.exp(cross_entropy)`. Мы также включаем метрику точности, чтобы увидеть, как часто наша модель правильно предсказывает следующее слово, поскольку кросс-энтропия (как мы уже видели) сложна для интерпретации и больше говорит о степени уверенности модели, чем о ее точности.

Вернемся к схеме процесса, показанной в начале этой главы. Первый этап уже выполнен и доступен в виде предварительно обученной модели в библиотеке fastai, а мы только что создали объекты `DataLoaders` и `Learner` для второго этапа. Теперь мы готовы выполнить тонкую настройку нашей языковой модели!

```markdown
<img alt="Схема процесса ULMFiT" width="450" src="images/att_00027.png">
```

Обучение каждой эпохи занимает довольно много времени, поэтому мы будем сохранять промежуточные результаты работы модели в процессе обучения. Поскольку функция `fine_tune` не делает этого автоматически, мы будем использовать `fit_one_cycle`. Как и в случае с `vision_learner`, `language_model_learner` автоматически вызывает функцию `freeze` при использовании предварительно обученной модели (что является настройкой по умолчанию), поэтому в этом случае будет обучаться только слой эмбеддингов (единственная часть модели, содержащая случайно инициализированные веса, то есть эмбеддинги для слов, которые есть в нашем словаре IMDb, но отсутствуют в словаре предварительно обученной модели):

In [ ]:
learn.fit_one_cycle(1, 2e-2)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,4.120048,3.912788,0.299565,50.038246,11:39


Эта модель требует длительного времени для обучения, поэтому это хороший повод поговорить о сохранении промежуточных результатов.

### Сохранение и загрузка моделей


Вы можете легко сохранить состояние вашей модели следующим образом:


In [ ]:
learn.save('1epoch')

Это создаст файл с именем *1epoch.pth* в директории `learn.path/models/`. Если вы хотите загрузить вашу модель на другом компьютере после создания объекта `Learner` или продолжить обучение позже, вы можете загрузить содержимое этого файла следующим образом:


In [ ]:
learn = learn.load('1epoch')

После завершения первоначального обучения мы можем продолжить точную настройку модели, "разморозив" ее:


In [ ]:
learn.unfreeze()
learn.fit_one_cycle(10, 2e-3)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,3.893486,3.772820,0.317104,43.502548,12:37
1,3.820479,3.717197,0.323790,41.148880,12:30
2,3.735622,3.659760,0.330321,38.851997,12:09
3,3.677086,3.624794,0.333960,37.516987,12:12
4,3.636646,3.601300,0.337017,36.645859,12:05
5,3.553636,3.584241,0.339355,36.026001,12:04
6,3.507634,3.571892,0.341353,35.583862,12:08
7,3.444101,3.565988,0.342194,35.374371,12:08
8,3.398597,3.566283,0.342647,35.384815,12:11
9,3.375563,3.568166,0.342528,35.451500,12:05


После этого мы сохраняем все части нашей модели, за исключением последнего слоя, который преобразует активации в вероятности выбора каждого токена из нашего словаря. Часть модели, не включающая этот последний слой, называется *кодировщиком*. Мы можем сохранить его с помощью функции `save_encoder`:


In [ ]:
learn.save_encoder('finetuned')

```markdown
> жаргон: Энкодер: Модель, не включающая финальный слой (или слои), специфичный для конкретной задачи. Этот термин имеет примерно то же значение, что и "базовая часть" ("body"), когда речь идет о сверточных нейронных сетях для обработки изображений, но термин "энкодер" чаще используется в области обработки естественного языка (NLP) и генеративных моделей.
```

Таким образом, завершается второй этап процесса классификации текста: тонкая настройка языковой модели. Теперь мы можем использовать ее для обучения классификатора, используя метки тональности из базы данных IMDb.

### Генерация текста


Прежде чем мы перейдем к тонкой настройке классификатора, давайте попробуем кое-что другое: используем нашу модель для генерации случайных отзывов. Поскольку модель обучена предсказывать следующее слово в предложении, мы можем использовать ее для написания новых отзывов:


In [ ]:
TEXT = "I liked this movie because"
N_WORDS = 40
N_SENTENCES = 2
preds = [learn.predict(TEXT, N_WORDS, temperature=0.75) 
         for _ in range(N_SENTENCES)]

In [ ]:
print("\n".join(preds))

i liked this movie because of its story and characters . The story line was very strong , very good for a sci - fi film . The main character , Alucard , was very well developed and brought the whole story
i liked this movie because i like the idea of the premise of the movie , the ( very ) convenient virus ( which , when you have to kill a few people , the " evil " machine has to be used to protect


Как вы видите, мы добавляем элемент случайности (мы выбираем случайное слово, основываясь на вероятностях, возвращенных моделью), чтобы не получать абсолютно одинаковые отзывы дважды. Наша модель не обладает запрограммированными знаниями о структуре предложений или правилах грамматики, но, тем не менее, она, очевидно, многому научилась об английских предложениях: мы видим, что она правильно использует заглавные буквы (*I* просто преобразуется в *i*, потому что наши правила требуют, чтобы слово состояло как минимум из двух символов, чтобы считаться написанным с заглавной буквы, поэтому вполне нормально, что оно встречается в нижнем регистре), и использует согласованное время. В целом, отзыв имеет смысл при первом прочтении, и только при внимательном чтении можно заметить, что что-то не совсем так. Неплохо для модели, обученной всего за несколько часов!

Но наша конечная цель заключалась не в том, чтобы обучить модель для генерации отзывов, а в том, чтобы классифицировать их... поэтому давайте использовать эту модель именно для этого.

### Создание объектов DataLoader для классификатора.

Мы переходим от тонкой настройки языковых моделей к тонкой настройке классификаторов. Для краткого повторения, языковая модель предсказывает следующее слово в тексте, поэтому ей не нужны внешние метки. Классификатор, напротив, предсказывает какую-либо внешнюю метку – в случае IMDb это, например, тональность текста.

Это означает, что структура нашего `DataBlock` для задач классификации в области обработки естественного языка будет очень знакомой. Она практически идентична той, что мы видели для многих наборов данных, используемых для классификации изображений:


In [ ]:
dls_clas = DataBlock(
    blocks=(TextBlock.from_folder(path, vocab=dls_lm.vocab),CategoryBlock),
    get_y = parent_label,
    get_items=partial(get_text_files, folders=['train', 'test']),
    splitter=GrandparentSplitter(valid_name='test')
).dataloaders(path, path=path, bs=128, seq_len=72)

Как и в случае классификации изображений, функция `show_batch` отображает зависимую переменную (в данном случае, эмоциональную окраску) для каждого значения независимой переменной (текста отзыва о фильме):

In [ ]:
dls_clas.show_batch(max_n=3)

,text,category
0,"xxbos i rate this movie with 3 skulls , only coz the girls knew how to scream , this could 've been a better movie , if actors were better , the twins were xxup ok , i believed they were evil , but the eldest and youngest brother , they sucked really bad , it seemed like they were reading the scripts instead of acting them … . spoiler : if they 're vampire 's why do they freeze the blood ? vampires ca n't drink frozen blood , the sister in the movie says let 's drink her while she is alive … .but then when they 're moving to another house , they take on a cooler they 're frozen blood . end of spoiler \n\n it was a huge waste of time , and that made me mad coz i read all the reviews of how",neg
1,"xxbos i have read all of the xxmaj love xxmaj come xxmaj softly books . xxmaj knowing full well that movies can not use all aspects of the book , but generally they at least have the main point of the book . i was highly disappointed in this movie . xxmaj the only thing that they have in this movie that is in the book is that xxmaj missy 's father comes to xxunk in the book both parents come ) . xxmaj that is all . xxmaj the story line was so twisted and far fetch and yes , sad , from the book , that i just could n't enjoy it . xxmaj even if i did n't read the book it was too sad . i do know that xxmaj pioneer life was rough , but the whole movie was a downer . xxmaj the rating",neg
2,"xxbos xxmaj this , for lack of a better term , movie is lousy . xxmaj where do i start … … \n\n xxmaj cinemaphotography - xxmaj this was , perhaps , the worst xxmaj i 've seen this year . xxmaj it looked like the camera was being tossed from camera man to camera man . xxmaj maybe they only had one camera . xxmaj it gives you the sensation of being a volleyball . \n\n xxmaj there are a bunch of scenes , haphazardly , thrown in with no continuity at all . xxmaj when they did the ' split screen ' , it was absurd . xxmaj everything was squished flat , it looked ridiculous . \n\n xxmaj the color tones were way off . xxmaj these people need to learn how to balance a camera . xxmaj this ' movie ' is poorly made , and",neg


Изучив определение `DataBlock`, можно заметить, что каждый его элемент знаком нам по предыдущим блокам данных, которые мы создавали, за исключением двух важных моментов:

- Параметр `is_lm=True` больше не присутствует в `TextBlock.from_folder`.
- Мы передаем `vocab` (словарь), который мы создали для тонкой настройки языковой модели.

Причина, по которой мы передаем `vocab` языковой модели, заключается в том, чтобы убедиться, что мы используем одинаковое соответствие между токенами и индексами. В противном случае, эмбеддинги, которые мы получили в процессе тонкой настройки языковой модели, не будут иметь смысла для этой модели, и этап тонкой настройки не принесет никакой пользы.

Указывая `is_lm=False` (или вообще не указывая `is_lm`, так как по умолчанию он равен `False`), мы сообщаем `TextBlock`, что у нас есть обычные размеченные данные, а не используем следующие токены в качестве меток. Однако, существует одна проблема, с которой нам необходимо справиться, а именно, объединение нескольких документов в мини-пакет. Давайте рассмотрим это на примере, попытавшись создать мини-пакет, содержащий первые 10 документов. Сначала мы преобразуем их в числовой вид:


In [ ]:
nums_samp = toks200[:10].map(num)

Теперь давайте посмотрим, сколько токенов содержит каждая из этих 10 рецензий на фильмы:

In [ ]:
nums_samp.map(len)

(#10) [228,238,121,290,196,194,533,124,581,155]

Помните, что `DataLoader` в PyTorch должен объединять все элементы в пакете в один тензор, а у одного тензора фиксированная форма (то есть, он имеет определенную длину по каждой оси, и все элементы должны быть согласованы). Это должно быть вам знакомо: у нас была аналогичная проблема с изображениями. В этом случае мы использовали обрезку, добавление отступов и/или сжатие, чтобы привести все входные данные к одному размеру. Обрезка может быть не лучшим вариантом для документов, поскольку, вероятно, мы удалим некоторую важную информацию (хотя та же проблема существует и для изображений, и мы используем обрезку там; расширение данных для задач обработки естественного языка еще недостаточно изучено, поэтому, возможно, есть возможности использовать обрезку и в этой области!). Вы не можете "сжать" документ. Таким образом, остается только добавление отступов!

Мы будем расширять самые короткие тексты, чтобы привести их все к одному размеру. Для этого мы используем специальный токен заполнения, который будет игнорироваться нашей моделью. Кроме того, чтобы избежать проблем с памятью и повысить производительность, мы будем объединять в пакеты тексты, имеющие примерно одинаковую длину (с некоторой рандомизацией для обучающего набора). Мы делаем это, (приблизительно, для обучающего набора), сортируя документы по длине перед каждой эпохой. В результате документы, объединенные в один пакет, будут иметь тенденцию к схожей длине. Мы не будем расширять каждый пакет до одного и того же размера, а будем использовать размер самого длинного документа в каждом пакете в качестве целевого размера. (Возможно сделать нечто подобное с изображениями, что особенно полезно для прямоугольных изображений, имеющих нерегулярные размеры, но на момент написания ни одна библиотека не предоставляет хорошей поддержки для этого, и нет никаких статей, посвященных этому. Однако, мы планируем добавить это в fastai в ближайшее время, поэтому следите за сайтом книги; мы добавим информацию об этом, как только это будет реализовано.)

Сортировка и добавление отступов автоматически выполняются API для работы с данными при использовании `TextBlock` с параметром `is_lm=False`. (У нас нет такой же проблемы с данными для языковых моделей, поскольку мы сначала объединяем все документы, а затем разделяем их на секции одинакового размера.)

Теперь мы можем создать модель для классификации наших текстов:


In [ ]:
learn = text_classifier_learner(dls_clas, AWD_LSTM, drop_mult=0.5, 
                                metrics=accuracy).to_fp16()

Финальный этап, предшествующий обучению классификатора, – это загрузка кодировщика из нашей предварительно обученной языковой модели. Мы используем функцию `load_encoder` вместо `load`, потому что у нас доступны только предварительно обученные веса для кодировщика; функция `load` по умолчанию вызывает исключение, если загружается неполная модель.

In [ ]:
learn = learn.load_encoder('finetuned')

### Тонкая настройка классификатора.

Последний этап – это обучение с использованием различных скоростей обучения и *постепенной разморозки* слоев. В области компьютерного зрения мы часто размораживаем всю модель сразу, но для классификаторов в области обработки естественного языка мы обнаружили, что размораживание нескольких слоев за раз оказывает существенное влияние:

In [ ]:
learn.fit_one_cycle(1, 2e-2)

epoch,train_loss,valid_loss,accuracy,time
0,0.347427,0.184480,0.929320,00:33


Всего за одну эпоху мы получаем тот же результат, что и при нашей обычной тренировке, описанной в разделе "<<chapter_intro>>": это неплохо! Мы можем передать значение `-2` параметру `freeze_to`, чтобы заморозить все слои, кроме последних двух групп параметров:

In [ ]:
learn.freeze_to(-2)
learn.fit_one_cycle(1, slice(1e-2/(2.6**4),1e-2))

epoch,train_loss,valid_loss,accuracy,time
0,0.247763,0.171683,0.934640,00:37


Затем мы можем немного ослабить ограничения и продолжить обучение:


In [ ]:
learn.freeze_to(-3)
learn.fit_one_cycle(1, slice(5e-3/(2.6**4),5e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.193377,0.156696,0.941200,00:45


И, наконец, вся модель целиком!

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(2, slice(1e-3/(2.6**4),1e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.172888,0.153770,0.943120,01:01
1,0.161492,0.155567,0.942640,00:57


Мы достигли точности 94,3%, что было передовым результатом всего три года назад. Обучив еще одну модель на всех текстах, прочитанных в обратном порядке, и усреднив прогнозы этих двух моделей, мы можем даже достичь точности 95,1%, что было наилучшим результатом, представленным в статье ULMFiT. Этот результат был превзойден лишь несколько месяцев назад, благодаря тонкой настройке гораздо более крупной модели и использованию дорогостоящих методов расширения данных (перевод предложений на другой язык и обратно, использование другой модели для перевода).

Использование предварительно обученной модели позволило нам создать тонко настроенную языковую модель, которая оказалась довольно мощной и может быть использована либо для создания поддельных отзывов, либо для их классификации. Это очень интересные разработки, но важно помнить, что эта технология также может быть использована в злонамеренных целях.

## Дезинформация и языковые модели.


Даже простые алгоритмы, основанные на правилах, существовавшие еще до появления широко доступных языковых моделей глубокого обучения, могли использоваться для создания фиктивных аккаунтов и попыток повлиять на политиков. Джефф Као, ныне журналист-аналитик в организации ProPublica, изучил комментарии, которые были отправлены в Федеральную комиссию по связи США (FCC) по поводу предложения 2017 года об отмене принципа сетевой нейтральности. В своей статье ["Более чем миллион комментариев в поддержку отмены сетевой нейтральности, вероятно, были сфальсифицированы"](https://hackernoon.com/more-than-a-million-pro-repeal-net-neutrality-comments-were-likely-faked-e9f0e3ed36a6), он рассказывает о том, как обнаружил большую группу комментариев, выступающих против сетевой нейтральности, которые, по-видимому, были сгенерированы с использованием метода, напоминающего заполнение шаблонов. В документах, посвященных дезинформации, Као полезно раскрасил эти поддельные комментарии разными цветами, чтобы подчеркнуть их шаблонный характер.

<img src="images/ethics/image16.png" width="700" id="disinformation" caption="Комментарии, полученные Федеральной комиссией по связи (FCC) в ходе дебатов о нейтралитете сети">

Компания Kao оценила, что "менее 800 000 из более чем 22 миллионов комментариев... можно считать действительно уникальными", и что "более 99% этих уникальных комментариев были в поддержку сохранения принципа сетевого нейтралитета".

Учитывая прогресс в области языкового моделирования, достигнутый с 2017 года, подобные мошеннические кампании теперь практически невозможно обнаружить. У вас есть все необходимые инструменты для создания эффективной языковой модели – то есть, системы, которая может генерировать текст, соответствующий контексту и выглядящий правдоподобно. Он не обязательно будет идеально точным или правильным, но будет выглядеть убедительно. Подумайте, что бы означало сочетание этой технологии с кампаниями дезинформации, о которых мы узнали в последние годы. Обратите внимание на диалог на Reddit, представленный в разделе <<ethics_reddit>>, где языковая модель, основанная на алгоритме OpenAI GPT-2, ведет диалог с самой собой о том, следует ли правительству США сократить расходы на оборону.

<img src="images/ethics/image14.png" id="ethics_reddit" caption="Алгоритм, общающийся сам с собой на Reddit" alt="Алгоритм, общающийся сам с собой на Reddit" width="600">

В данном случае было прямо указано, что использовался определенный алгоритм, но представьте, что произошло бы, если бы недобросовестный человек решил распространить подобный алгоритм в социальных сетях. Он мог бы делать это постепенно и осторожно, позволяя алгоритму со временем приобретать последователей и завоевывать доверие. Для создания буквально миллионов аккаунтов, занимающихся этим, потребовалось бы относительно немного ресурсов. В такой ситуации легко представить себе ситуацию, когда подавляющее большинство онлайн-дискуссий велось бы ботами, и никто бы даже не подозревал об этом.

Мы уже начинаем видеть примеры использования машинного обучения для создания фальшивых личностей. Например, <<katie_jones>> демонстрирует профиль в LinkedIn, принадлежащий якобы Кэти Джонс.

<img src="images/ethics/image15.jpeg" width="400" id="katie_jones" caption="Профиль Кэти Джонс в LinkedIn">

Кейти Джонс была связана в LinkedIn с несколькими сотрудниками ведущих аналитических центров Вашингтона. Но ее не существовало. То изображение, которое вы видите, было сгенерировано нейронной сетью, и человек под именем Кейти Джонс, на самом деле, не окончила Центр стратегических и международных исследований.

Многие люди предполагают или надеются, что алгоритмы придут нам на помощь в этой ситуации, что мы разработаем алгоритмы классификации, которые смогут автоматически распознавать контент, сгенерированный искусственным интеллектом. Однако проблема заключается в том, что это всегда будет своего рода гонка вооружений, в которой более совершенные алгоритмы классификации (или дискриминации) могут использоваться для создания более продвинутых алгоритмов генерации.

## Заключение


В этой главе мы рассмотрели последнее приложение, которое из коробки поддерживается библиотекой fastai: обработка текста. Мы увидели два типа моделей: языковые модели, способные генерировать текст, и классификатор, определяющий, является ли отзыв положительным или отрицательным. Для создания высокопроизводительного классификатора мы использовали предварительно обученную языковую модель, затем адаптировали ее к набору данных нашей задачи, а после этого использовали ее основную часть (кодировщик) с новым выходным слоем для выполнения классификации.

Прежде чем завершить этот раздел, давайте рассмотрим, как библиотека fastai может помочь вам подготовить данные для решения конкретных задач.

## Анкета


1. Что такое "самообучение"?
2. Что такое "языковая модель"?
3. Почему языковая модель считается моделью, обучаемой с самоконтролем?
4. Для каких целей обычно используются модели, обучаемые с самоконтролем?
5. Почему мы дообучаем языковые модели?
6. Какие три шага необходимы для создания современного классификатора текста?
7. Как 50 000 неразмеченных отзывов о фильмах помогают нам создать более совершенный классификатор текста для набора данных IMDb?
8. Какие три шага необходимо выполнить для подготовки данных для языковой модели?
9. Что такое "токенизация"? Зачем она нужна?
10. Назовите три различных подхода к токенизации.
11. Что такое `xxbos`?
12. Перечислите четыре правила, которые fastai применяет к тексту во время токенизации.
13. Почему повторяющиеся символы заменяются токеном, показывающим количество повторений и символ, который повторяется?
14. Что такое "численное представление"?
15. Почему некоторые слова могут быть заменены токеном "неизвестное слово"?
16. Если размер пакета равен 64, первая строка тензора, представляющего первый пакет, содержит первые 64 токена для набора данных. Что содержит вторая строка этого тензора? Что содержит первая строка второго пакета? (Будьте внимательны — студенты часто ошибаются в этом вопросе! Обязательно проверьте свой ответ на веб-сайте книги.)
17. Зачем нам нужна "подкладка" (padding) для классификации текста? Почему она не нужна для языкового моделирования?
18. Что содержит матрица представлений (embedding matrix) для обработки естественного языка (NLP)? Какова ее форма?
19. Что такое "перплексия"?
20. Почему мы должны передавать словарный запас языковой модели в блок данных для классификатора?
21. Что такое "постепенная разморозка"?
22. Почему генерация текста всегда, вероятно, будет опережать автоматическое распознавание текстов, созданных машиной?

### Дальнейшие исследования


1. Узнайте, что можно узнать о языковых моделях и дезинформации. Какие сегодня являются лучшими языковыми моделями? Ознакомьтесь с некоторыми примерами их работы. Считаете ли вы их убедительными? Как недобросовестный человек может использовать такую модель для создания конфликтов и неопределенности?
1. Учитывая ограничение, заключающееся в том, что модели, вероятно, не смогут постоянно распознавать тексты, созданные машиной, какие другие подходы могут потребоваться для борьбы с масштабными кампаниями дезинформации, использующими глубокое обучение?